In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import timm

from ImageModelUtils import OrderFlowRegressor, RelativeHuberLoss, RelativeMSELoss
from ImageEncoding import OrderFlowDataset, ToImage

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'
DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/'
LOCAL_DATA_DIR = '/content/data'

!cp -r {DATA_DIR} {LOCAL_DATA_DIR}

Mounted at /content/drive


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

img_transform = ToImage(output_size=(600, 160, 3))
train_dataset = OrderFlowDataset(
    f"{LOCAL_DATA_DIR}/train.csv", 
    f"{LOCAL_DATA_DIR}/book_train.parquet", 
    f"{LOCAL_DATA_DIR}/trade_train.parquet", 
    transform=img_transform
)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

/content/ImageEncoding.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.books = {k : v for k, v in full_book.groupby(['stock_id', 'time_id'])}
/content/ImageEncoding.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.trades = {k : v for k, v in full_trade.groupby(['stock_id', 'time_id'])}


In [3]:
hyperparameters = {
    "batch_size": 512, 
    "num_workers": os.cpu_count() - 1 if os.cpu_count() else 0, 
    "pin_memory": True, 
    "prefetch_factor": 2
}

In [4]:
def main():
    train_loader = DataLoader(train_dataset, shuffle=True, **hyperparameters)
    val_loader = DataLoader(val_dataset, shuffle=False, **hyperparameters)
    model = OrderFlowRegressor().to(device)
    criterion = RelativeMSELoss()

    muon_params = [p for p in model.parameters() if p.ndim == 2]
    adamw_params = [p for p in model.parameters() if p.ndim != 2]

    optimizer_adamw = optim.AdamW(adamw_params, lr=3e-5, weight_decay=0.05)
    optimizer_muon = optim.Muon(muon_params, lr=0.001, momentum=0.95)

    num_epochs = 30
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.1 * total_steps)
    decay_steps = total_steps - warmup_steps

    def build_scheduler(optimizer):
        warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
        cosine = CosineAnnealingLR(optimizer, T_max=decay_steps)
        return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    scheduler_adamw = build_scheduler(optimizer_adamw)
    scheduler_muon = build_scheduler(optimizer_muon)

    checkpoint_path = f"{DIR}/checkpoint.pth"
    start_epoch = 0
    best_val_loss = float('inf')
    train_losses = []
    test_losses = []

    if os.path.exists(checkpoint_path):
        print(f"Found checkpoint at {checkpoint_path}. Resuming training...")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer_adamw.load_state_dict(checkpoint['optimizer_adamw_state_dict'])
        optimizer_muon.load_state_dict(checkpoint['optimizer_muon_state_dict'])
        scheduler_adamw.load_state_dict(checkpoint['scheduler_adamw_state_dict'])
        scheduler_muon.load_state_dict(checkpoint['scheduler_muon_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        train_losses = checkpoint['train_losses']
        test_losses = checkpoint['val_losses']
        print(f"Resuming from epoch {start_epoch + 1}...")

    try:
        for epoch in range(start_epoch, num_epochs):
            model.train()
            running_loss = 0.0
            
            for batch in train_loader:
                # Extract tensors from the dictionary
                images = batch["image"]
                targets = batch["r_vol"]
                
                # 1. Rearrange from (B, H, W, C) to (B, C, H, W) -> [512, 3, 600, 160]
                images = images.permute(0, 3, 1, 2)
                    
                # 2. Cast from np.int32 to float32 and move to GPU
                images = images.to(device, dtype=torch.float32, non_blocking=True)
                targets = torch.as_tensor(targets, dtype=torch.float32, device=device)
                
                optimizer_adamw.zero_grad()
                optimizer_muon.zero_grad()
                
                with autocast('cuda', dtype=torch.bfloat16):
                    outputs = model(images)
                
                loss = criterion(outputs.float(), targets)
                loss.backward()    
                
                optimizer_adamw.step()
                optimizer_muon.step()
                
                scheduler_adamw.step()
                scheduler_muon.step()
                
                running_loss += loss.item() * images.size(0)
                
            epoch_loss = running_loss / len(train_dataset)
            train_losses.append(epoch_loss)
            print(f"Epoch {epoch+1}/{num_epochs} | Training Loss: {epoch_loss:.6f}")

            model.eval()
            running_val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader: 
                    images = batch["image"]
                    targets = batch["r_vol"]
                    
                    images = images.permute(0, 3, 1, 2)
                        
                    images = images.to(device, dtype=torch.float32, non_blocking=True)
                    targets = torch.as_tensor(targets, dtype=torch.float32, device = device)
                    
                    with autocast('cuda', dtype=torch.bfloat16):
                        outputs = model(images)
                        val_loss = criterion(outputs, targets)
                        
                    running_val_loss += val_loss.item() * images.size(0)

            val_loss = running_val_loss / len(val_dataset)
            test_losses.append(val_loss)
            print(f"Epoch {epoch+1}/{num_epochs} | Validation Loss: {val_loss:.6f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f"{DIR}/best_model.pth")
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_adamw_state_dict': optimizer_adamw.state_dict(),
                'optimizer_muon_state_dict': optimizer_muon.state_dict(),
                'scheduler_adamw_state_dict': scheduler_adamw.state_dict(),
                'scheduler_muon_state_dict': scheduler_muon.state_dict(),
                'best_val_loss': best_val_loss,
                'train_losses': train_losses,
                'val_losses': test_losses
            }, checkpoint_path)
    except KeyboardInterrupt:
        print("Training interrupted. Saving checkpoint...")
    finally:
        if train_losses:
            min_len = min(len(train_losses), len(test_losses))
            history_df = pd.DataFrame({
                'epoch': list(range(1, min_len + 1)),
                'train_loss': train_losses[:min_len],
                'val_loss': test_losses[:min_len]
            })
            history_df.to_csv(f"{DIR}/training_history.csv", index=False)

In [5]:
if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Epoch 1/30 | Training Loss: 431.488850
Epoch 1/30 | Validation Loss: 222.878297
Epoch 2/30 | Training Loss: 180.967208
Epoch 2/30 | Validation Loss: 133.813757
Epoch 3/30 | Training Loss: 140.199883
Epoch 3/30 | Validation Loss: 139.820555
Epoch 4/30 | Training Loss: 95.354207
Epoch 4/30 | Validation Loss: 74.566964
Epoch 5/30 | Training Loss: 46.937645
Epoch 5/30 | Validation Loss: 32.892917
Epoch 6/30 | Training Loss: 26.171310
Epoch 6/30 | Validation Loss: 17.991760
Epoch 7/30 | Training Loss: 16.822968
Epoch 7/30 | Validation Loss: 0.253722
Epoch 8/30 | Training Loss: 6.534691
Epoch 8/30 | Validation Loss: 3.677083
Epoch 9/30 | Training Loss: 2.021162
Epoch 9/30 | Validation Loss: 0.769495
Epoch 10/30 | Training Loss: 0.403550
Epoch 10/30 | Validation Loss: 0.277269
Epoch 11/30 | Training Loss: 0.280447
Epoch 11/30 | Validation Loss: 0.388280
Epoch 12/30 | Training Loss: 0.259172
Epoch 12/30 | Validation Loss: 0.296946
Epoch 13/30 | Training Loss: 0.236386
Epoch 13/30 | Validation 